In [0]:
%run ../fw

In [0]:
dbutils.widgets.text("pipeline_name","")
pipeline_name = dbutils.widgets.get("pipeline_name")

In [0]:
conf = (
    spark.table("retail_sales_dw.config_table")
    .filter(col("pipeline_name") == pipeline_name)
    .first()
    )

In [0]:
s = SilverLayer(
    table_name=conf.table_name,
    schema_detail=conf.schema_detail,
    keys=conf.keys,
    write_mode=conf.write_mode
)

bronze_df = s.read_add_sk_from_bronze_table()

invalid_df = s.get_invalid_record(bronze_df)

key_null_df = s.get_key_null_record(bronze_df)

dup_df = s.get_dup_record(
    bronze_df,
    key_null_df
)

all_bad_df = s.get_all_bad_record(
    invalid_df,
    key_null_df,
    dup_df
)

final_result_df = s.get_final_result(
    bronze_df,
    all_bad_df
)

s.load_bad_record(all_bad_df)

s.load_to_silver_layer(final_result_df)

In [0]:
print("Bronze:", bronze_df.count())
print("Invalid:", invalid_df.count())
print("Null key:", key_null_df.count())
print("Duplicate:", dup_df.count())
print("All bad:", all_bad_df.count())
print("Final:", final_result_df.count())